# Pipeline Comparison Experiment
## BM25 vs Hybrid (BM25 + Dense + RRF + Cohere Rerank) vs GraphRAG

This notebook runs all three retrieval pipelines on the **val set** (10 English queries with gold citations)
and compares them across Macro F1, Precision, Recall, MAP, NDCG@10, and per-query latency.

### What each pipeline actually does
All three modes share the same **deterministic decompose-loop** in `llm.define_agent.run_agent`
(this replaced the old ReAct agent):

1. **Decompose** the query into focused legal sub-issues, each with precise German search terms
   (`llm.decompose.decompose_query`).
2. **Search** the relevant tool(s) per sub-issue (`search_laws` / `search_courts`) and union all
   returned citations into an order-preserving candidate pool.
3. **Sibling-consideration expansion** (court decisions): once any consideration of a decision is
   retrieved, add all its sibling considerations (`expand.expand_court_siblings`).

The only thing that varies between modes is the **search backend**:
- **BM25** — `LawSearchTool` / `CourtSearchTool` (multilingual BM25, CombMAX fusion over EN/DE/FR/IT).
- **Hybrid** — `HybridSearchTool` = BM25 + dense (local `multilingual-e5-large`) → RRF → Cohere rerank.
- **GraphRAG** — `GraphRAGLocalSearchTool` wrapped to fit the same tool interface.

> **Note on the metrics.** `run_agent` returns the *full candidate pool* with no final relevance-selection
> step, so it is recall-oriented: expect high recall but low precision (hence low Macro F1). The rank-aware
> metrics (MAP, NDCG@10) and **Recall** are the most informative for comparing retrieval backends here.

### Prerequisites
- `data/val.csv`, `data/laws_de.csv`, `data/court_considerations.csv` present
- `.env` at repo root with `API_KEY` (required for all modes — drives decomposition + translation)
- For **Hybrid**: a local embedding model at `models/multilingual-e5-large` (preferred) **or** `API_KEY`
  for the academic-cloud embedding fallback; optional `COHERE_API_KEY` for reranking
- For **GraphRAG**: a pre-built index (`graphrag/output/*.parquet`, via `scripts/run_graphrag_index.sh`)

Results are cached to `output/experiment_cache/` so re-running the notebook skips LLM calls.

---
## 1. Environment Setup

In [ ]:
import os
import sys
import json
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Resolve repo root regardless of where the kernel started
cwd = Path.cwd()
REPO_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
os.chdir(REPO_ROOT)

# Put pipeline modules on the path
for p in [str(REPO_ROOT / "src" / "our_pipeline"), str(REPO_ROOT / "src")]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"Repo root : {REPO_ROOT}")
print(f"Python    : {sys.version.split()[0]}")

In [ ]:
from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

API_KEY    = os.getenv("API_KEY", "")
COHERE_KEY = os.getenv("COHERE_API_KEY", "")

# Hybrid uses a LOCAL embedding model when present, else the academic-cloud API (API_KEY).
LOCAL_EMBED_DIR = REPO_ROOT / "models" / "multilingual-e5-large"
LOCAL_EMBED_OK  = LOCAL_EMBED_DIR.is_dir()

print(f"API_KEY         : {'set' if API_KEY else 'MISSING — required for all modes'}")
print(f"COHERE_API_KEY  : {'set' if COHERE_KEY else 'not set — Hybrid will skip Cohere rerank'}")
print(f"Local embeddings: {'found at ' + str(LOCAL_EMBED_DIR) if LOCAL_EMBED_OK else 'not found — Hybrid will use API embeddings (needs API_KEY)'}")

---
## 2. Experiment Configuration

Toggle modes and set shared parameters here.

In [ ]:
EXP = {
    # ── Which modes to run ────────────────────────────────────────────────
    "run_bm25"    : True,
    "run_hybrid"  : True,   # needs local embedding model OR API_KEY
    "run_graphrag": True,   # needs pre-built graphrag index

    # ── Shared retrieval settings ─────────────────────────────────────────
    "top_k_laws"            : 20,    # results per law sub-issue search
    "top_k_courts"          : 20,    # results per court sub-issue search
    "max_decompose_issues"  : 12,    # max focused sub-issues per query
    "enable_sibling_expansion": True,  # add sibling considerations of retrieved court decisions

    # ── Dense index (Hybrid only) ─────────────────────────────────────────
    "max_courts_dense"   : 200_000,  # limit court dense index size

    # ── Cache ─────────────────────────────────────────────────────────────
    "use_cache"  : True,
    "cache_dir"  : REPO_ROOT / "output" / "experiment_cache",

    # ── Plotting ──────────────────────────────────────────────────────────
    "fig_dpi"    : 120,
    "palette"    : {"BM25": "#4C72B0", "Hybrid": "#DD8452", "GraphRAG": "#55A868"},
}

EXP["cache_dir"].mkdir(parents=True, exist_ok=True)
print("Config ready.")

---
## 3. Data Loading

In [ ]:
import pandas as pd
import numpy as np

val_path = REPO_ROOT / "data" / "val.csv"
assert val_path.exists(), f"val.csv not found at {val_path}"

val_df = pd.read_csv(val_path)
print(f"Loaded {len(val_df)} queries from val.csv")
val_df[["query_id", "query"]].head()

---
## 4. Build Shared BM25 Indices

Both BM25 and Hybrid share the same BM25 index objects — built once and reused.

In [ ]:
from corpus import get_or_build_index, get_or_build_dense_index
from constants import (
    LAWS_CSV, COURTS_CSV,
    LAWS_INDEX_PATH, COURTS_INDEX_PATH,
    LAWS_DENSE_INDEX_DIR, COURTS_DENSE_INDEX_DIR,
    CONFIG,
)

laws_bm25   = get_or_build_index("laws",   LAWS_CSV,   LAWS_INDEX_PATH)
courts_bm25 = get_or_build_index("courts", COURTS_CSV, COURTS_INDEX_PATH)

print(f"Laws index  : {len(laws_bm25.documents):,} documents")
print(f"Courts index: {len(courts_bm25.documents):,} documents")

# Court sibling-consideration index (decision -> all its considerations).
# run_agent uses this to expand decision-level hits into exact-consideration hits.
from expand import build_court_sibling_index

sibling_index = None
if EXP["enable_sibling_expansion"]:
    sibling_index = build_court_sibling_index(courts_bm25.documents)
    print(f"Sibling index: {len(sibling_index):,} court decisions "
          f"from {len(courts_bm25.documents):,} considerations")

---
## 5. Helper Functions

In [ ]:
from omnilex.evaluation.metrics import (
    citation_f1, macro_f1, micro_f1,
    mean_average_precision, mean_ndcg_at_k,
)


def parse_citations(s) -> list[str]:
    """Split semicolon-delimited citation string into a list."""
    if not s or str(s).strip() in ("", "nan"):
        return []
    return [c.strip() for c in str(s).split(";") if c.strip()]


def query_metrics(predicted: list[str], gold: list[str]) -> dict:
    pred_set, gold_set = set(predicted), set(gold)
    tp = len(pred_set & gold_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)
    scores = citation_f1(predicted, gold)
    return {**scores, "tp": tp, "fp": fp, "fn": fn,
            "n_predicted": len(pred_set), "n_gold": len(gold_set)}


def load_cache(mode: str):
    p = EXP["cache_dir"] / f"{mode}.json"
    return json.loads(p.read_text()) if p.exists() else None


def save_cache(mode: str, data: dict):
    p = EXP["cache_dir"] / f"{mode}.json"
    p.write_text(json.dumps(data, indent=2))


def run_experiment(tools: dict, mode_label: str, sibling_index: dict | None = None) -> dict:
    """Run the decompose-loop agent over val.csv with given tools; return structured results."""
    if EXP["use_cache"]:
        cached = load_cache(mode_label)
        if cached:
            print(f"[{mode_label}] Loaded {len(cached['queries'])} results from cache.")
            return cached

    from llm.define_agent import run_agent

    # Apply shared decompose-loop settings read by run_agent / decompose_query at call time.
    # (Per-tool top_k is set on the tool objects themselves, not via CONFIG.)
    CONFIG["max_decompose_issues"]    = EXP["max_decompose_issues"]
    CONFIG["enable_sibling_expansion"] = EXP["enable_sibling_expansion"]

    query_results = []
    for _, row in val_df.iterrows():
        qid   = row["query_id"]
        query = row["query"]
        gold  = parse_citations(row.get("gold_citations", ""))

        t0 = time.time()
        predicted, _ = run_agent(query, tools=tools, verbose=False,
                                 sibling_index=sibling_index)
        elapsed = round(time.time() - t0, 2)

        m = query_metrics(predicted, gold)
        query_results.append({
            "query_id": qid, "query": query,
            "predicted": predicted, "gold": gold,
            "elapsed_secs": elapsed, **m,
        })
        print(f"  {qid}  F1={m['f1']:.3f}  recall={m['recall']:.3f}  "
              f"(pred={m['n_predicted']} gold={m['n_gold']} "
              f"TP={m['tp']} FP={m['fp']} FN={m['fn']}  {elapsed}s)")

    all_preds = [r["predicted"] for r in query_results]
    all_gold  = [r["gold"]      for r in query_results]
    total_t   = sum(r["elapsed_secs"] for r in query_results)

    result = {
        "mode": mode_label,
        "queries": query_results,
        "metrics": {
            **macro_f1(all_preds, all_gold),
            **micro_f1(all_preds, all_gold),
            "map":          mean_average_precision(all_preds, all_gold),
            "ndcg_at_10":   mean_ndcg_at_k(all_preds, all_gold, k=10),
            "total_elapsed_secs": round(total_t, 2),
            "avg_elapsed_secs":   round(total_t / len(query_results), 2),
        },
    }

    if EXP["use_cache"]:
        save_cache(mode_label, result)

    return result


print("Helpers ready.")

---
## 6. Experiment A — BM25 Only

In [ ]:
results_bm25 = None

if EXP["run_bm25"]:
    from search_tools import LawSearchTool as PL, CourtSearchTool as PC

    bm25_tools = {
        "search_laws":   PL(laws_bm25,   top_k=EXP["top_k_laws"]),
        "search_courts": PC(courts_bm25, top_k=EXP["top_k_courts"]),
    }

    print("Running BM25 experiment...")
    results_bm25 = run_experiment(bm25_tools, "BM25", sibling_index=sibling_index)
    print(f"\nBM25 Macro F1 : {results_bm25['metrics']['macro_f1']:.4f}")
    print(f"BM25 Recall   : {results_bm25['metrics']['macro_recall']:.4f}")
    print(f"BM25 MAP      : {results_bm25['metrics']['map']:.4f}")
else:
    print("BM25 experiment skipped (run_bm25=False)")

---
## 7. Experiment B — Hybrid (BM25 + Dense + RRF + Cohere Rerank)

In [ ]:
results_hybrid = None

if EXP["run_hybrid"]:
    # Embedding backend: prefer the local multilingual-e5-large model; otherwise
    # fall back to the academic-cloud embeddings API (needs API_KEY). Mirrors
    # the logic in src/our_pipeline/pipeline_test.py.
    if LOCAL_EMBED_OK:
        _embed_model  = str(LOCAL_EMBED_DIR)
        _embed_apikey = None
        _embed_base   = None
        print(f"Embedding backend: local ({LOCAL_EMBED_DIR})")
    elif API_KEY:
        _embed_model  = CONFIG.get("embedding_model", "e5-mistral-7b-instruct")
        # If embedding_model points at a (missing) local dir, use the API default instead.
        if "/" in _embed_model and not (REPO_ROOT / _embed_model).is_dir():
            _embed_model = "e5-mistral-7b-instruct"
        _embed_apikey = API_KEY
        _embed_base   = CONFIG.get("embedding_api_base", "https://chat-ai.academiccloud.de/v1")
        print(f"Embedding backend: API ({_embed_base}, model={_embed_model})")
    else:
        _embed_model = None

    if _embed_model is None:
        print("Hybrid SKIPPED: no local embedding model and API_KEY not set.")
        print(f"  Expected local model at: {LOCAL_EMBED_DIR}")
    else:
        print("Building dense indices (loads from cache if already built)...")
        laws_dense = get_or_build_dense_index(
            "laws", LAWS_CSV, LAWS_DENSE_INDEX_DIR,
            max_rows=CONFIG.get("max_docs_dense_laws"),
            model=_embed_model,
            api_key=_embed_apikey,
            api_base=_embed_base or "https://chat-ai.academiccloud.de/v1",
            batch_size=CONFIG.get("embedding_batch_size", 32),
        )
        courts_dense = get_or_build_dense_index(
            "courts", COURTS_CSV, COURTS_DENSE_INDEX_DIR,
            max_rows=EXP["max_courts_dense"],
            model=_embed_model,
            api_key=_embed_apikey,
            api_base=_embed_base or "https://chat-ai.academiccloud.de/v1",
            batch_size=CONFIG.get("embedding_batch_size", 32),
        )

        from search_tools import HybridSearchTool

        # HybridSearchTool reads COHERE_API_KEY from the environment for reranking.
        if COHERE_KEY:
            os.environ["COHERE_API_KEY"] = COHERE_KEY
        else:
            print("Note: COHERE_API_KEY not set — Hybrid will use RRF results without reranking.")

        hybrid_tools = {
            "search_laws":   HybridSearchTool(laws_bm25,   laws_dense,   "laws",   top_k=EXP["top_k_laws"]),
            "search_courts": HybridSearchTool(courts_bm25, courts_dense, "courts", top_k=EXP["top_k_courts"]),
        }

        print("\nRunning Hybrid experiment...")
        results_hybrid = run_experiment(hybrid_tools, "Hybrid", sibling_index=sibling_index)
        print(f"\nHybrid Macro F1 : {results_hybrid['metrics']['macro_f1']:.4f}")
        print(f"Hybrid Recall   : {results_hybrid['metrics']['macro_recall']:.4f}")
        print(f"Hybrid MAP      : {results_hybrid['metrics']['map']:.4f}")
else:
    print("Hybrid experiment skipped (run_hybrid=False)")

---
## 8. Experiment C — GraphRAG

Requires a completed knowledge graph index (`graphrag/output/*.parquet`).
Build it with `sbatch scripts/run_graphrag_index.sh` (or `graphrag index --root ./graphrag`).

The `GraphRAGLocalSearchTool` is wrapped in a thin adapter so it plugs into the same
`run_agent` decompose-loop as the BM25 and Hybrid tools (it exposes `get_last_results()`
returning citation dicts). Citations are parsed from the GraphRAG answer text via regex.

In [ ]:
import re as _re

results_graphrag = None
GRAPHRAG_ROOT    = REPO_ROOT / "graphrag"

# GraphRAG index needs the full set of output tables the local-search API loads.
_REQUIRED_PARQUETS = ["entities", "communities", "community_reports", "text_units", "relationships"]
_missing = [f for f in _REQUIRED_PARQUETS if not (GRAPHRAG_ROOT / "output" / f"{f}.parquet").exists()]
graphrag_ready = not _missing

if EXP["run_graphrag"] and not graphrag_ready:
    print(f"GraphRAG SKIPPED: index incomplete (missing: {_missing}).")
    print("Build it with:  sbatch scripts/run_graphrag_index.sh   (or: graphrag index --root ./graphrag)")

elif EXP["run_graphrag"] and graphrag_ready:
    from omnilex.retrieval.graphrag_tools import GraphRAGLocalSearchTool

    class _GraphRAGAdapter:
        """Wrap GraphRAG local search so it matches the run_agent tool interface.

        run_agent calls ``tool(query)`` then reads ``tool.get_last_results()`` —
        a list of ``{"citation": ..., "text": ..., "_score": ...}`` dicts — so this
        adapter parses citations out of the GraphRAG answer text into that shape.
        """

        _CITATION_PATTERNS = [
            r"Art\.\s*\d+[a-z]?\s*(?:Abs\.\s*\d+\s+)?[A-Z]{2,5}",
            r"BGE\s+\d{1,3}\s+[IVX]+[a-z]?\s+\d+(?:\s+E\.?\s*[\d.]+)?",
            r"\d[A-Z]_\d+/\d{4}\s+E\.?\s*[\d.]+",
        ]

        def __init__(self, local_tool, corpus_label: str):
            self._tool        = local_tool
            self.name         = f"search_{corpus_label}"
            self.corpus_label = corpus_label
            self.description  = (
                f"Search {corpus_label} via the GraphRAG knowledge graph. "
                "Input: a legal question. Output: an answer with citations."
            )
            self._last_results: list[dict] = []

        def __call__(self, query: str) -> str:
            return self.run(query)

        def run(self, query: str) -> str:
            if not query or not query.strip():
                self._last_results = []
                return "Error: Empty query."
            response = str(self._tool(query))
            self._last_results = [
                {"citation": c, "text": "", "_score": None}
                for c in self._extract(response)
            ]
            return response

        def _extract(self, text: str) -> list[str]:
            found: list[str] = []
            for pat in self._CITATION_PATTERNS:
                found.extend(_re.findall(pat, text))
            # de-duplicate while preserving order
            seen, out = set(), []
            for c in found:
                c = c.strip()
                if c and c not in seen:
                    seen.add(c)
                    out.append(c)
            return out

        def get_last_results(self) -> list[dict]:
            return self._last_results

        def get_last_citations(self) -> list[str]:
            return [d["citation"] for d in self._last_results]

    _local = GraphRAGLocalSearchTool(graphrag_root=GRAPHRAG_ROOT)

    graphrag_tools = {
        "search_laws":   _GraphRAGAdapter(_local, "laws"),
        "search_courts": _GraphRAGAdapter(_local, "courts"),
    }

    print("Running GraphRAG experiment...")
    results_graphrag = run_experiment(graphrag_tools, "GraphRAG", sibling_index=sibling_index)
    print(f"\nGraphRAG Macro F1 : {results_graphrag['metrics']['macro_f1']:.4f}")
    print(f"GraphRAG Recall   : {results_graphrag['metrics']['macro_recall']:.4f}")
    print(f"GraphRAG MAP      : {results_graphrag['metrics']['map']:.4f}")
else:
    print("GraphRAG experiment skipped (run_graphrag=False)")

---
## 9. Compile Results

In [ ]:
all_results: dict[str, dict] = {
    label: res
    for label, res in [
        ("BM25",    results_bm25),
        ("Hybrid",  results_hybrid),
        ("GraphRAG",results_graphrag),
    ]
    if res is not None
}

if not all_results:
    raise RuntimeError("No experiments completed — check config and .env file.")

modes = list(all_results.keys())
colors = [EXP["palette"].get(m, "#888") for m in modes]
print(f"Comparing modes: {modes}")

# ── Aggregate metrics table ────────────────────────────────────────────────
summary_df = pd.DataFrame([
    {
        "Mode":             m,
        "Macro F1":         r["metrics"]["macro_f1"],
        "Macro Precision":  r["metrics"]["macro_precision"],
        "Macro Recall":     r["metrics"]["macro_recall"],
        "Micro F1":         r["metrics"]["micro_f1"],
        "MAP":              r["metrics"]["map"],
        "NDCG@10":          r["metrics"]["ndcg_at_10"],
        "Avg Time (s)":     r["metrics"]["avg_elapsed_secs"],
    }
    for m, r in all_results.items()
]).set_index("Mode")

print("\nAggregate metrics:")
display(summary_df.round(4).style.highlight_max(axis=0, color="#d4edda").highlight_min(axis=0, color="#f8d7da"))

# ── Per-query long-form DataFrame ─────────────────────────────────────────
pq_rows = []
for mode, data in all_results.items():
    for q in data["queries"]:
        pq_rows.append({
            "mode":         mode,
            "query_id":     q["query_id"],
            "f1":           q["f1"],
            "precision":    q["precision"],
            "recall":       q["recall"],
            "tp":           q["tp"],
            "fp":           q["fp"],
            "fn":           q["fn"],
            "n_predicted":  q["n_predicted"],
            "n_gold":       q["n_gold"],
            "elapsed_secs": q["elapsed_secs"],
        })

pq_df = pd.DataFrame(pq_rows)

---
## 10. Visualisations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker

plt.rcParams.update({
    "figure.dpi":       EXP["fig_dpi"],
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.grid":        True,
    "grid.alpha":       0.35,
    "axes.titlesize":   11,
    "axes.labelsize":   10,
    "font.family":      "DejaVu Sans",
})

PLOT_DIR = REPO_ROOT / "output" / "experiment_plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Plots will be saved to: {PLOT_DIR}")

### Plot 1 — Aggregate Metrics Comparison

In [ ]:
metric_cols = ["Macro F1", "Macro Precision", "Macro Recall", "Micro F1", "MAP", "NDCG@10"]
n_metrics = len(metric_cols)
n_modes   = len(modes)
x         = np.arange(n_metrics)
width      = 0.8 / n_modes

fig, ax = plt.subplots(figsize=(11, 5))

for i, (mode, color) in enumerate(zip(modes, colors)):
    vals   = [summary_df.loc[mode, c] for c in metric_cols]
    offset = (i - (n_modes - 1) / 2) * width
    bars   = ax.bar(x + offset, vals, width=width * 0.92,
                    color=color, label=mode, edgecolor="white", linewidth=0.6)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{v:.3f}", ha="center", va="bottom", fontsize=7.5, color="#333")

ax.set_xticks(x)
ax.set_xticklabels(metric_cols, fontsize=9)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Score")
ax.set_title("Aggregate Retrieval Metrics by Pipeline Mode", fontweight="bold", pad=10)
ax.legend(framealpha=0.9, fontsize=9)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))

fig.tight_layout()
fig.savefig(PLOT_DIR / "01_metrics_comparison.png", bbox_inches="tight")
plt.show()
print("Saved: 01_metrics_comparison.png")

### Plot 2 — Per-Query F1 Heatmap

In [ ]:
query_ids = sorted(pq_df["query_id"].unique())
heatmap_data = np.full((len(modes), len(query_ids)), np.nan)

for ri, mode in enumerate(modes):
    for ci, qid in enumerate(query_ids):
        row = pq_df[(pq_df["mode"] == mode) & (pq_df["query_id"] == qid)]
        if not row.empty:
            heatmap_data[ri, ci] = row["f1"].values[0]

fig, ax = plt.subplots(figsize=(max(8, len(query_ids) * 0.9), len(modes) * 1.0 + 1.2))

im = ax.imshow(heatmap_data, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)

ax.set_xticks(range(len(query_ids)))
ax.set_xticklabels(query_ids, rotation=35, ha="right", fontsize=8)
ax.set_yticks(range(len(modes)))
ax.set_yticklabels(modes, fontsize=9)
ax.set_title("Per-Query F1 Score by Pipeline Mode", fontweight="bold", pad=10)

for ri in range(len(modes)):
    for ci in range(len(query_ids)):
        v = heatmap_data[ri, ci]
        if not np.isnan(v):
            text_color = "white" if v < 0.4 or v > 0.75 else "black"
            ax.text(ci, ri, f"{v:.2f}", ha="center", va="center",
                    fontsize=8.5, color=text_color, fontweight="bold")

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
cbar.set_label("F1 Score", fontsize=9)

fig.tight_layout()
fig.savefig(PLOT_DIR / "02_per_query_f1_heatmap.png", bbox_inches="tight")
plt.show()
print("Saved: 02_per_query_f1_heatmap.png")

### Plot 3 — Precision vs Recall Scatter

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))

for mode, color in zip(modes, colors):
    sub = pq_df[pq_df["mode"] == mode]
    ax.scatter(sub["recall"], sub["precision"],
               color=color, label=mode, s=90, alpha=0.85,
               edgecolors="white", linewidths=0.8, zorder=3)
    # mode centroid
    ax.scatter(sub["recall"].mean(), sub["precision"].mean(),
               color=color, marker="D", s=160,
               edgecolors="black", linewidths=1.0, zorder=4)

# iso-F1 curves
for f1_val in [0.2, 0.4, 0.6, 0.8]:
    r_vals = np.linspace(0.01, 1.0, 300)
    p_vals = f1_val * r_vals / (2 * r_vals - f1_val)
    mask   = (p_vals >= 0) & (p_vals <= 1)
    ax.plot(r_vals[mask], p_vals[mask], color="#bbb",
            linewidth=0.8, linestyle="--", zorder=1)
    if mask.any():
        mid = mask.nonzero()[0][len(mask.nonzero()[0]) // 2]
        ax.text(r_vals[mid] + 0.01, p_vals[mid], f"F1={f1_val}",
                fontsize=7, color="#999")

ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.15)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision vs Recall per Query\n(diamonds = mode centroid)",
             fontweight="bold")
ax.legend(fontsize=9, framealpha=0.9)

fig.tight_layout()
fig.savefig(PLOT_DIR / "03_precision_recall_scatter.png", bbox_inches="tight")
plt.show()
print("Saved: 03_precision_recall_scatter.png")

### Plot 4 — F1 Distribution (Strip + Box)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))

for i, (mode, color) in enumerate(zip(modes, colors)):
    vals = pq_df[pq_df["mode"] == mode]["f1"].values

    # box
    bp = ax.boxplot(
        vals, positions=[i], widths=0.4,
        patch_artist=True, notch=False,
        boxprops=dict(facecolor=color, alpha=0.35, linewidth=1.2),
        medianprops=dict(color=color, linewidth=2.2),
        whiskerprops=dict(color="#777", linewidth=1.1),
        capprops=dict(color="#777", linewidth=1.1),
        flierprops=dict(marker="o", color=color, alpha=0.5, markersize=4),
    )

    # individual points (jittered)
    jitter = np.random.default_rng(42).uniform(-0.08, 0.08, size=len(vals))
    ax.scatter(np.full(len(vals), i) + jitter, vals,
               color=color, s=55, alpha=0.85, zorder=3,
               edgecolors="white", linewidths=0.6)

    # mean label
    ax.text(i, vals.mean() + 0.04, f"μ={vals.mean():.3f}",
            ha="center", fontsize=8, color=color, fontweight="bold")

ax.set_xticks(range(len(modes)))
ax.set_xticklabels(modes, fontsize=10)
ax.set_ylim(-0.05, 1.2)
ax.set_ylabel("F1 Score (per query)")
ax.set_title("Distribution of Per-Query F1 Scores", fontweight="bold")

fig.tight_layout()
fig.savefig(PLOT_DIR / "04_f1_distribution.png", bbox_inches="tight")
plt.show()
print("Saved: 04_f1_distribution.png")

### Plot 5 — TP / FP / FN Breakdown

In [ ]:
tp_fp_fn = (
    pq_df.groupby("mode")[["tp", "fp", "fn"]]
    .sum()
    .reindex(modes)
)

fig, ax = plt.subplots(figsize=(7, len(modes) * 0.9 + 1.5))

bar_h    = 0.55
y_pos    = np.arange(len(modes))
tp_vals  = tp_fp_fn["tp"].values
fp_vals  = tp_fp_fn["fp"].values
fn_vals  = tp_fp_fn["fn"].values

b1 = ax.barh(y_pos, tp_vals, height=bar_h, color="#2ca02c", label="TP (correct)")
b2 = ax.barh(y_pos, fp_vals, height=bar_h, left=tp_vals,
             color="#ff7f0e", label="FP (wrong predictions)")
b3 = ax.barh(y_pos, fn_vals, height=bar_h, left=tp_vals + fp_vals,
             color="#d62728", label="FN (missed citations)")

def label_bars(bars, offset=0):
    for bar in bars:
        w = bar.get_width()
        if w > 0:
            ax.text(bar.get_x() + w / 2 + offset, bar.get_y() + bar.get_height() / 2,
                    str(int(w)), ha="center", va="center",
                    fontsize=9, color="white", fontweight="bold")

label_bars(b1)
label_bars(b2)
label_bars(b3)

ax.set_yticks(y_pos)
ax.set_yticklabels(modes, fontsize=10)
ax.set_xlabel("Total citations (summed across all queries)")
ax.set_title("TP / FP / FN Breakdown by Pipeline Mode", fontweight="bold")
ax.legend(loc="lower right", fontsize=9, framealpha=0.9)
ax.invert_yaxis()

fig.tight_layout()
fig.savefig(PLOT_DIR / "05_tp_fp_fn_breakdown.png", bbox_inches="tight")
plt.show()
print("Saved: 05_tp_fp_fn_breakdown.png")

### Plot 6 — Average Query Latency

In [ ]:
avg_times = [all_results[m]["metrics"]["avg_elapsed_secs"] for m in modes]

fig, ax = plt.subplots(figsize=(5.5, 4))

bars = ax.bar(modes, avg_times, color=colors, edgecolor="white", linewidth=0.8)

for bar, t in zip(bars, avg_times):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(avg_times) * 0.01,
            f"{t:.1f}s", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_ylabel("Average time per query (seconds)")
ax.set_title("Per-Query Latency by Pipeline Mode", fontweight="bold")
ax.set_ylim(0, max(avg_times) * 1.2)

fig.tight_layout()
fig.savefig(PLOT_DIR / "06_latency_comparison.png", bbox_inches="tight")
plt.show()
print("Saved: 06_latency_comparison.png")

### Plot 7 — Per-Query Prediction vs Gold Citation Count

In [ ]:
query_ids_sorted = sorted(pq_df["query_id"].unique())
x = np.arange(len(query_ids_sorted))
width = 0.75 / (n_modes + 1)  # +1 for gold bar

fig, ax = plt.subplots(figsize=(11, 4.5))

# gold counts (same per query regardless of mode)
gold_counts = [
    pq_df[pq_df["query_id"] == qid]["n_gold"].values[0]
    for qid in query_ids_sorted
]
offset_gold = (-(n_modes) / 2) * width
ax.bar(x + offset_gold, gold_counts, width=width * 0.92,
       color="#888", label="Gold", edgecolor="white", linewidth=0.6, alpha=0.7)

for i, (mode, color) in enumerate(zip(modes, colors)):
    pred_counts = [
        pq_df[(pq_df["mode"] == mode) & (pq_df["query_id"] == qid)]["n_predicted"].values[0]
        if not pq_df[(pq_df["mode"] == mode) & (pq_df["query_id"] == qid)].empty else 0
        for qid in query_ids_sorted
    ]
    offset = (i - (n_modes - 1) / 2 + 0.5) * width
    ax.bar(x + offset, pred_counts, width=width * 0.92,
           color=color, label=mode, edgecolor="white", linewidth=0.6, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(query_ids_sorted, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("Citation count")
ax.set_title("Predicted vs Gold Citation Count per Query", fontweight="bold")
ax.legend(fontsize=9, framealpha=0.9)

fig.tight_layout()
fig.savefig(PLOT_DIR / "07_citation_counts.png", bbox_inches="tight")
plt.show()
print("Saved: 07_citation_counts.png")

### Plot 8 — Radar / Spider Chart of Aggregate Metrics

In [ ]:
radar_metrics  = ["Macro F1", "Macro Precision", "Macro Recall", "Micro F1", "MAP", "NDCG@10"]
n_radar        = len(radar_metrics)
angles         = np.linspace(0, 2 * np.pi, n_radar, endpoint=False).tolist()
angles        += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

for mode, color in zip(modes, colors):
    vals = [summary_df.loc[mode, m] for m in radar_metrics]
    vals += vals[:1]
    ax.plot(angles, vals, color=color, linewidth=2, label=mode)
    ax.fill(angles, vals, color=color, alpha=0.12)

ax.set_thetagrids(np.degrees(angles[:-1]), radar_metrics, fontsize=9)
ax.set_ylim(0, 1)
ax.yaxis.set_tick_params(labelsize=7)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=7)
ax.set_title("Radar: Aggregate Metrics per Mode", fontweight="bold",
             pad=20, fontsize=11)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=9, framealpha=0.9)

fig.tight_layout()
fig.savefig(PLOT_DIR / "08_radar_chart.png", bbox_inches="tight")
plt.show()
print("Saved: 08_radar_chart.png")

---
## 11. Summary Table & Observations

In [ ]:
best_mode_macro_f1 = summary_df["Macro F1"].idxmax()
best_mode_map      = summary_df["MAP"].idxmax()
fastest_mode       = summary_df["Avg Time (s)"].idxmin()

print("=" * 55)
print(" EXPERIMENT SUMMARY")
print("=" * 55)
print(f" Queries evaluated : {len(val_df)}")
print(f" Modes compared    : {', '.join(modes)}")
print("-" * 55)
print(f" Best Macro F1     : {best_mode_macro_f1}  "
      f"({summary_df.loc[best_mode_macro_f1, 'Macro F1']:.4f})")
print(f" Best MAP          : {best_mode_map}  "
      f"({summary_df.loc[best_mode_map, 'MAP']:.4f})")
print(f" Fastest mode      : {fastest_mode}  "
      f"({summary_df.loc[fastest_mode, 'Avg Time (s)']:.1f}s / query)")
print("=" * 55)

print("\nFull metrics table:")
display(summary_df.round(4))

print(f"\nPlots saved to: {PLOT_DIR}")

In [ ]:
# Export summary to CSV alongside the plots
summary_df.round(4).to_csv(PLOT_DIR / "summary_metrics.csv")
pq_df.to_csv(PLOT_DIR / "per_query_results.csv", index=False)
print("Exported summary_metrics.csv and per_query_results.csv")